# C10_E01 — Z–N, Lambda y SIMC sobre el mismo FOPDT

**Capítulo:** 10 — Sintonía de PID y diagnóstico de lazos  
**Identificador:** `C10_E01`  
**Objetivo pedagógico:** Comparar tres métodos de sintonía industrial sobre la misma planta.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Sintonía de un lazo de temperatura de horno: comparación con criterio IAE.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Tres métodos de sintonía sobre el mismo FOPDT

In [1]:
import numpy as np
import control as ct
import matplotlib.pyplot as plt
import os

K, tau, theta = 0.8, 600.0, 60.0    # FOPDT identificado de un horno

def zn_pi(K, tau, theta):
    """Ziegler-Nichols por curva de reacción para PI."""
    Kc = 0.9*tau/(K*theta); Ti = 3.33*theta
    return Kc, Ti

def lambda_pi(K, tau, theta, lam):
    Kc = tau/(K*(lam + theta)); Ti = tau
    return Kc, Ti

def simc_pi(K, tau, theta, lam=None):
    if lam is None: lam = theta
    Kc = (1.0/K)*tau/(lam + theta); Ti = min(tau, 4*(lam + theta))
    return Kc, Ti

zn   = zn_pi  (K, tau, theta)
lamb = lambda_pi(K, tau, theta, lam=theta)
simc = simc_pi(K, tau, theta)
print(f"Z-N:    Kc={zn[0]:.3f}, Ti={zn[1]:.1f}")
print(f"Lambda: Kc={lamb[0]:.3f}, Ti={lamb[1]:.1f}")
print(f"SIMC:   Kc={simc[0]:.3f}, Ti={simc[1]:.1f}")

Z-N:    Kc=11.250, Ti=199.8
Lambda: Kc=6.250, Ti=600.0
SIMC:   Kc=6.250, Ti=480.0


## 2. Simulación y comparativa

In [2]:
# Aproximación por Pade del retardo (orden 1) para simular en s
def pade1(theta):
    return ct.tf([-theta/2.0, 1.0], [theta/2.0, 1.0])

G = ct.tf([K], [tau, 1.0])*pade1(theta)
t = np.linspace(0, 4000, 1500)

fig, ax = plt.subplots(figsize=(8, 4))
for label, (Kc, Ti) in [("Z-N", zn), ("Lambda", lamb), ("SIMC", simc)]:
    PI = ct.tf([Kc*Ti, Kc], [Ti, 0.0])
    sysC = ct.feedback(PI*G, 1)
    _, y = ct.step_response(sysC, t)
    iae = np.trapz(np.abs(1 - y), t)
    ax.plot(t, y, label=f"{label} (IAE={iae:.0f})")
ax.axhline(1.0, ls='--', color='gray')
ax.set_xlabel("t (s)"); ax.set_ylabel("y / SP")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C10_E01 — Z-N vs Lambda vs SIMC sobre el mismo FOPDT")
out = '../../figures/cap10/fig_C10_F01_sintonias.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

/sessions/laughing-tender-dirac/tmp/ipykernel_7/846023690.py:13: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  iae = np.trapz(np.abs(1 - y), t)


## 3. Interpretación

Para este FOPDT con θ/τ = 0.1, Z-N produce una sintonía agresiva (mayor sobredisparo, mayor IAE en seguimiento), mientras que Lambda y SIMC producen sintonías más conservadoras y robustas. **No usar Z-N como método final en procesos con θ/τ alto:** el lazo entra en régimen de margen de fase pequeño y se inestabiliza ante variaciones paramétricas o de carga.